# Evaluation of 2D Embeddings

In [ ]:
%load_ext autoreload
%autoreload 2

import io
import cairosvg
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import KDTree

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

from dual_ifm.classification.eval_finetune import load_embeddings
from dual_ifm.interpretation.eval import get_KNN_metrics, get_2d_PCA
from dual_ifm.utils import datasets, plot

plot.set_rc_params(kind='paper', notebook_dpi=120)
pd.options.display.float_format = '{:,.3f}'.format
device = 'cuda:0'

## Example 1 - DR classification with the APTOS dataset

In [ ]:
dataset_dir = './datasets'
prefix = 'tsimcneb'
dataset_name = 'aptos'
feature_name = 'dr'
kfold = 3
img_size = (256, 256)  # Size of the input to the model
sample_size = None
sweep_name = f'clf_{prefix}_{dataset_name}_{feature_name}_{img_size[0]}'
experiment_name = f'dataset.kfold={kfold}_dataset={dataset_name}'

project_dir = Path.cwd()
checkpoints_dir = project_dir.joinpath('checkpoints')

In [ ]:
X, X_2d, y = load_embeddings(checkpoints_dir.joinpath(sweep_name), experiment_name)

dataset, mapping = datasets.load_dataset(
    dataset_dir=dataset_dir,
    dataset_name=dataset_name,
    transform=None,
    image_size=img_size,
    feature_name=feature_name,
    drop_nan=False,
    sample_size=None,
    split='all_holdout',
)
y = np.expand_dims(y, axis=-1)

### Visualize 2D embeddings

In [ ]:
n_subplots = (1, 1)
fig_width = 'full'
fig_height_ratio = 0.6
imbalanced = [False]
categorical = [False]

# For visualization purposes (comment if using another checkpoint)
rot = plot.get_rotation_matrix(60)
X_2d_rot= X_2d @ rot
mask = (X_2d_rot[:, 0] > -20) & (X_2d_rot[:, 0] < 20) & (X_2d_rot[:, 1] > -25) & (X_2d_rot[:, 1] < 25)
X_2d_filt = X_2d_rot[mask]
y_filt = y[mask]

fig, ax = plot.plot_embeddings(
    X_2d_filt,
    y_filt,
    [mapping],
    None,
    n_subplots,
    fig_width,
    fig_height_ratio,
    [''],
    imbalanced,
    categorical,
    cbar_shrink=0.2,
    cbar_location='bottom',
    s_marker=12,
    return_ax=True,
)
ax = ax[0]

### Add images to the embeddings
1. Nearest neighbors

In [ ]:
# Select an image and find NNs
kdtree = KDTree(X_2d_filt)
neighbors_distances, neighbors_indices = kdtree.query([0, -20], k=20)

for neighbor_idx in neighbors_indices:
    print(neighbor_idx, dataset[neighbor_idx][1])

In [ ]:
images, embedding_coords, image_labels = [], [], []
for neighbors_idx in neighbors_indices:
    image, label = dataset[neighbors_idx]

    images.append(image)
    embedding_coords.append(X_2d_filt[neighbors_idx])
    image_labels.append(label)

In [ ]:
# Annotate figure with images
img_coords = [(-25, 20), (-25, 0), (-25, -20), (20, 20), (20, 0)]
image_titles = [str(int(l)) for l in image_labels]
fig_neigh, ax_neigh = plot.annotate_embeddings_plot(fig, ax, images, image_labels, image_titles, embedding_coords, img_coords)
fig_neigh

2. From all classses

In [ ]:
image_indices = [693, 2126, 15, 235, 1030]

images, embedding_coords, image_labels = [], [], []
for image_idx in image_indices:
    image, label = dataset[image_idx]

    images.append(image)
    embedding_coords.append(X_2d_filt[image_idx])
    image_labels.append(label)

In [ ]:
# Annotate figure with images
label_map = {0: '0 - Healthy', 1: '1 - Mild', 2: '2 - Moderate', 3: '3 - Severe', 4: '4 - Proliferative'}

img_coords = [(-25, 20), (-25, -0), (-25, -20), (15, 20), (15, 0)]
image_titles = [label_map[l] for l in image_labels]
fig_classes, ax_classes = plot.annotate_embeddings_plot(fig, ax, images, image_labels, image_titles, embedding_coords, img_coords)

ax.set_title('A. APTOS', fontweight='bold', fontsize=12)
fig.subplots_adjust(top=0.95)

# Adjust colorbar location, colorbar axes is the last one added
cbar_ax = fig.axes[-1]
pos = cbar_ax.get_position()
cbar_ax.set_position([pos.x0 + 0.05, pos.y0, pos.width, pos.height])

plt.tight_layout()
fig.savefig('./plots/2d_embeddings_aptos.pdf')
fig.savefig('./plots/2d_embeddings_aptos.svg')
fig_classes

## Example 2 - Glaucoma classification with the Glaucoma dataset

In [ ]:
prefix = 'tsimcneb'
dataset_name = 'glaucoma'
feature_name = 'glaucoma'
kfold = 5
img_size = (256, 256)  # Size of the input to the model
sweep_name = f'clf_{prefix}_{dataset_name}_{feature_name}_{img_size[0]}'
experiment_name = f'dataset.kfold={kfold}_dataset={dataset_name}'

project_dir = Path.cwd()
checkpoints_dir = project_dir.joinpath('checkpoints')

In [ ]:
X, X_2d, y = load_embeddings(checkpoints_dir.joinpath(sweep_name), experiment_name)

dataset, mapping = datasets.load_dataset(
    dataset_dir=dataset_dir,
    dataset_name=dataset_name,
    transform=None,
    image_size=img_size,
    feature_name=feature_name,
    drop_nan=False,
    sample_size=None,
    split='all_holdout',
)
y = np.expand_dims(y, axis=-1)

In [ ]:
n_subplots = (1, 1)
fig_width = 'full'
fig_height_ratio = 0.6
imbalanced = [False]
categorical = [False]

# For visualization purposes (comment if using another checkpoint)
rot = plot.get_rotation_matrix(-30)
X_2d_rot= X_2d @ rot
mask = (X_2d_rot[:, 1] > -700) & (X_2d_rot[:, 1] < 700)
X_2d_filt = X_2d_rot[mask]
y_filt = y[mask]

fig, ax = plot.plot_embeddings(
    X_2d_filt,
    y_filt,
    [mapping],
    None,
    n_subplots,
    fig_width,
    fig_height_ratio,
    [''],
    imbalanced,
    categorical,
    cbar_shrink=0.2,
    cbar_location='bottom',
    s_marker=30,
    return_ax=True,
)
ax = ax[0]

In [ ]:
# Select an image and find NNs
kdtree = KDTree(X_2d_filt)

neighbors_distances, neighbors_indices = kdtree.query([25, -10], k=5)
print(neighbors_indices, y[neighbors_indices].flatten())

In [ ]:
for neighbor_idx in neighbors_indices:
    print(neighbor_idx, dataset[neighbor_idx][1])

### Add images to the embeddings

In [ ]:
image_indices = [945, 1440, 178]

images, embedding_coords, image_labels = [], [], []
for image_idx in image_indices:
    image, label = dataset[image_idx]

    images.append(image)
    embedding_coords.append(X_2d_filt[image_idx])
    image_labels.append(label)

In [ ]:
# Annotate figure with image
label_map = {0: '0 - Normal', 1: '1 - Early', 2: '2 - Advanced'}

img_coords = [(60, 65), (60, 25), (60, -15)]
image_titles = [label_map[l] for l in image_labels]
fig_classes, ax_classes = plot.annotate_embeddings_plot(fig, ax, images, image_labels, image_titles, embedding_coords, img_coords)

ax.set_title('B. Glaucoma Fundus', fontweight='bold', fontsize=12)
fig.subplots_adjust(top=0.95)

plt.tight_layout()
fig.savefig('./plots/2d_embeddings_glaucoma.pdf')
fig.savefig('./plots/2d_embeddings_glaucoma.svg')
fig_classes

## KNN accuracy

In [ ]:
backbone_names = ['BagNet33']
weight_names = ['ImageNet', 'SimCLR', 't-SimCNE' , 't-SimCNE aligned', 't-SimCNEx' , 't-SimCNEx aligned']  #
# dataset_names = ['EyePACS', 'AREDS', 'UKB'] # In pretraining
dataset_names = ['APTOS', 'DeepDRiD', 'IDRiD', 'Messidor', 'Glaucoma', 'PAPILA', 'FIVES']

df_rows = []
for dataset_name in dataset_names:
    if dataset_name in ['AREDS']:
        feature_name = 'amd'
    elif dataset_name in ['Glaucoma', 'PAPILA']:
        feature_name = 'glaucoma'
    elif dataset_name in ['FIVES']:
        feature_name = 'disease'
    else:
        feature_name = 'dr'

    for backbone_name in backbone_names:
        for weight_name in weight_names:
            # Experiment name
            prefix = f'{weight_name.split(sep=' ')[0].replace("-", "").lower()}{backbone_name[0].lower()}'

            if 'aligned' in weight_name:
                sweep_name = f'clfproj_{prefix}_{dataset_name.lower()}_{feature_name}_256'
            else:
                sweep_name = f'clf_{prefix}_{dataset_name.lower()}_{feature_name}_256'

            for fold in range(1, 6):
                experiment_name = f'dataset.kfold={fold}_dataset={dataset_name.lower()}'

                if os.path.exists(checkpoints_dir.joinpath(sweep_name, experiment_name + '_embeddings.json')):
                    X, X_2d, y = load_embeddings(checkpoints_dir.joinpath(sweep_name), experiment_name)
                    auroc, auprc, acc, kappa = get_KNN_metrics(X_2d, y, n_neighbors=10)
                    row = {
                        'Model': f'{backbone_name} {weight_name}',
                        'Dataset': dataset_name,
                        'kfold': fold,
                        'AUROC': auroc,
                        'AUPRC': auprc,
                        'Balanced Accuracy': acc,
                        'Kappa': kappa,
                    }
                    df_rows.append(row)

df = pd.DataFrame(df_rows)

In [ ]:
df_groupby = df.drop(columns='kfold').groupby(by=['Dataset', 'Model']).agg(('mean', 'std'))
df_groupby

In [ ]:
# Display one metric
df_metric = df_groupby['AUROC'].copy()
df_metric['mean_sd'] = df_metric['mean'].round(3).astype(str)
table_metric = df_metric['mean_sd'].reset_index().pivot(index='Dataset', columns='Model', values='mean_sd')
table_metric = table_metric.reindex(dataset_names)
table_metric

In [ ]:
# Display one metric + sd
df_metric = df_groupby['AUROC'].copy()
df_metric['mean_sd'] = df_metric['mean'].round(3).astype(str) + ' ± ' + df_metric['std'].round(3).astype(str)
table_metric = df_metric['mean_sd'].reset_index().pivot(index='Dataset', columns='Model', values='mean_sd')
table_metric = table_metric.reindex(dataset_names)
table_metric

In [ ]:
print(table_metric.to_csv(sep='\t'))

In [ ]:
latex_table = table_metric.to_latex(
    float_format='%.3f',
    multicolumn=True,
    multirow=True,
    caption='caption',
    label='tab:label',
)

print(latex_table)

### PCA

In [ ]:
backbone_names = ['BagNet33']
weight_names = ['ImageNet', 'SimCLR', 't-SimCNE' , 't-SimCNE aligned', 't-SimCNEx' , 't-SimCNEx aligned']   #
# dataset_names = ['EyePACS', 'AREDS', 'UKB'] # In pretraining
dataset_names = ['APTOS', 'DeepDRiD', 'IDRiD', 'Messidor', 'Glaucoma', 'PAPILA', 'FIVES']

df_rows = []
for dataset_name in dataset_names:
    if dataset_name in ['AREDS']:
        feature_name = 'amd'
    elif dataset_name in ['Glaucoma', 'PAPILA']:
        feature_name = 'glaucoma'
    elif dataset_name in ['FIVES']:
        feature_name = 'disease'
    else:
        feature_name = 'dr'

    for backbone_name in backbone_names:
        for weight_name in weight_names:
            # Experiment name
            prefix = f'{weight_name.split(sep=' ')[0].replace("-", "").lower()}{backbone_name[0].lower()}'

            if 'aligned' in weight_name:
                sweep_name = f'clfproj_{prefix}_{dataset_name.lower()}_{feature_name}_256'
            else:
                sweep_name = f'clf_{prefix}_{dataset_name.lower()}_{feature_name}_256'

            for fold in range(1, 6):
                experiment_name = f'dataset.kfold={fold}_dataset={dataset_name.lower()}'
                
                if os.path.exists(checkpoints_dir.joinpath(sweep_name, experiment_name + '_embeddings.json')):
                    X, X_2d, y = load_embeddings(checkpoints_dir.joinpath(sweep_name), experiment_name)

                    # 2d pca for other models than tsimcne
                    if 't-simcne' not in weight_name.lower():
                        X_2d = get_2d_PCA(X)

                    auroc, auprc, acc, kappa = get_KNN_metrics(X_2d, y, n_neighbors=10)
                    row = {
                        'Model': f'{backbone_name} {weight_name}',
                        'Dataset': dataset_name,
                        'kfold': fold,
                        'AUROC': auroc,
                        'AUPRC': auprc,
                        'Balanced Accuracy': acc,
                        'Kappa': kappa,
                    }
                    df_rows.append(row)

df_pca = pd.DataFrame(df_rows)

In [ ]:
df_groupby = df_pca.drop(columns='kfold').groupby(by=['Dataset', 'Model']).agg(('mean', 'std'))
df_groupby

In [ ]:
# Display one metric
df_metric = df_groupby['AUROC'].copy()
df_metric['mean_sd'] = df_metric['mean'].round(3).astype(str)
table_metric = df_metric['mean_sd'].reset_index().pivot(index='Dataset', columns='Model', values='mean_sd')
table_metric = table_metric.reindex(dataset_names)
table_metric

In [ ]:
# Display one metric + sd
df_metric = df_groupby['AUROC'].copy()
df_metric['mean_sd'] = df_metric['mean'].round(3).astype(str) + ' ± ' + df_metric['std'].round(3).astype(str)
table_metric = df_metric['mean_sd'].reset_index().pivot(index='Dataset', columns='Model', values='mean_sd')
table_metric = table_metric.reindex(dataset_names)
table_metric

In [ ]:
print(table_metric.to_csv(sep='\t'))

### Plot all embeddings from a dataset

In [ ]:
datasets = ['idrid', 'aptos', 'messidor', 'deepdrid', 'glaucoma', 'papila', 'fives', 'eyepacs', 'areds']
features = feature_name = {'areds': 'amd', 'glaucoma': 'glaucoma', 'papila': 'glaucoma', 'fives': 'disease'}

for dataset_name in datasets:
    prefix = 'tsimcnexb'
    feature_name = features.get(dataset_name, 'dr')
    img_size = (256, 256)
    sweep_name = f'clfproj_{prefix}_{dataset_name}_{feature_name}_{img_size[0]}'

    fig, axes = plt.subplots(2, 3)
    plot.set_figsize(fig, 'full', height_ratio=0.55)

    for i, ax in enumerate(axes.flatten()[:5]):
        experiment_name = f'dataset.kfold={i + 1}_dataset={dataset_name}'
        svg_path = checkpoints_dir / sweep_name / f'{experiment_name}_embeddings.svg'
        img = plt.imread(io.BytesIO(cairosvg.svg2png(url=str(svg_path), dpi=300)))
        ax.imshow(img)
        ax.axis('off')

    axes.flatten()[-1].set_visible(False)
    plt.tight_layout()
    fig.savefig(f'./plots/{dataset_name}.png', dpi=300)
    plt.close(fig)